In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/dataset_ekstrak/PlantVillage"

In [ ]:
import zipfile
import os

# Menentukan path file zip asal dan folder tujuan ekstraksi
zip_path = "/content/drive/MyDrive/proyek_agrikultur/archive (2).zip"
extract_path = "/content/dataset_ekstrak"

print("Sedang mengekstrak dataset, mohon tunggu...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Ekstraksi selesai! Folder baru siap digunakan di:", extract_path)

Sedang mengekstrak dataset, mohon tunggu...
Ekstraksi selesai! Folder baru siap digunakan di: /content/dataset_ekstrak


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import os

# =====================================================================
# 1. KONFIGURASI PATH DATASET (Mengarah ke subfolder PlantVillage)
# =====================================================================
DATASET_PATH = "/content/dataset_ekstrak/PlantVillage"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 10

# =====================================================================
# 2. PRA-PEMROSESAN & AUGMENTASI DATA
# =====================================================================
print("--- Mempersiapkan dan Memproses Dataset ---")

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

num_classes = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())
print(f"\nBerhasil memuat {num_classes} kelas penyakit tanaman:")
for idx, name in enumerate(class_names):
    print(f" Kelas {idx}: {name}")

# =====================================================================
# 3. MEMBANGUN ARSITEKTUR MODEL CNN (Sudah Diperbaiki Utan)
# =====================================================================
print("\n--- Membangun Arsitektur Model CNN ---")

model = models.Sequential([
    # Layer Konvolusi 1 & Pooling
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D((2, 2)),

    # Layer Konvolusi 2 & Pooling
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Layer Konvolusi 3 & Pooling
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Layer Konvolusi 4 & Pooling
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Meratakan matriks menjadi vektor 1D
    layers.Flatten(),

    # Fully Connected Layer
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    # Output Layer sesuai jumlah penyakit tanaman
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =====================================================================
# 4. PROSES PELATIHAN MODEL (TRAINING)
# =====================================================================
print("\n--- Memulai Proses Training Model AI ---")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator
)

# =====================================================================
# 5. MENYIMPAN MODEL TERLATIH KE GOOGLE DRIVE
# =====================================================================
print("\n--- Menyimpan Model AI ---")
backup_save_path = '/content/drive/MyDrive/model_agrikultur_presisi.keras'

try:
    model.save(backup_save_path)
    print(f"Selesai! Otak AI Anda sukses dilatih dan disimpan di Drive dengan path:\n{backup_save_path}")
except Exception as e:
    print(f"Gagal menyimpan ke Drive secara otomatis. Error: {e}")

--- Mempersiapkan dan Memproses Dataset ---
Found 16516 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.

Berhasil memuat 15 kelas penyakit tanaman:
 Kelas 0: Pepper__bell___Bacterial_spot
 Kelas 1: Pepper__bell___healthy
 Kelas 2: Potato___Early_blight
 Kelas 3: Potato___Late_blight
 Kelas 4: Potato___healthy
 Kelas 5: Tomato_Bacterial_spot
 Kelas 6: Tomato_Early_blight
 Kelas 7: Tomato_Late_blight
 Kelas 8: Tomato_Leaf_Mold
 Kelas 9: Tomato_Septoria_leaf_spot
 Kelas 10: Tomato_Spider_mites_Two_spotted_spider_mite
 Kelas 11: Tomato__Target_Spot
 Kelas 12: Tomato__Tomato_YellowLeaf__Curl_Virus
 Kelas 13: Tomato__Tomato_mosaic_virus
 Kelas 14: Tomato_healthy

--- Membangun Arsitektur Model CNN ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,605,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,850,575 (7.06 MB)

 Trainable params: 1,850,575 (7.06 MB)

 Non-trainable params: 0 (0.00 B)


--- Memulai Proses Training Model AI ---
Epoch 1/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 158s 292ms/step - accuracy: 0.3809 - loss: 1.8937 - val_accuracy: 0.5956 - val_loss: 1.2257
Epoch 2/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 129s 250ms/step - accuracy: 0.6307 - loss: 1.1029 - val_accuracy: 0.7496 - val_loss: 0.7385
Epoch 3/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 130s 252ms/step - accuracy: 0.7155 - loss: 0.8347 - val_accuracy: 0.7506 - val_loss: 0.7280
Epoch 4/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 129s 249ms/step - accuracy: 0.7663 - loss: 0.6927 - val_accuracy: 0.8047 - val_loss: 0.5798
Epoch 5/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 127s 245ms/step - accuracy: 0.7955 - loss: 0.6101 - val_accuracy: 0.8134 - val_loss: 0.5287
Epoch 6/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 128s 247ms/step - accuracy: 0.8265 - loss: 0.5065 - val_accuracy: 0.7763 - val_loss: 0.6801
Epoch 7/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 128s 248ms/step - accuracy: 0.8383 - loss: 0.4694 - val_accuracy: 0.8768 - val_loss: 0.3585
Epoch 8/10
517/517 ━━━━━━━━━━━━━━━